# Excel: просадка ЧОД и Финреза в марте 2026

Отдельная диагностика **только по Excel-отчётам** Jan–Jun 2026.

Цель: понять, почему в **марте** просели `ЧОД` и `Фин. Рез.` относительно соседних месяцев.

## Что делает
1. Грузит Excel `01_Январь` … `06_Июнь_2026.xlsx`.
2. Сводит помесячные тоталы: ЧОД, Финрез, комиссии, АУР, амортизация, term/trx.
3. Считает просадку марта vs среднее остальных месяцев и vs Feb/Apr.
4. На зерне `inn+agr_id`: TOP agr по падению ЧОД/Финреза March−Feb и March−Apr.
5. Проверяет тождество `Финрез ≈ ЧОД − АУР − Амортизация`.

Lake/`final_df` не обязателен (опциональная сверка в конце, если есть CSV).


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUTPUT_DIR = DATA_DIR
FOCUS_MONTH = '2026-03'
PEER_MONTHS = ['2026-01', '2026-02', '2026-04', '2026-05', '2026-06']

excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-03': 0,
    '2026-04': 0,
    '2026-05': 0,
    '2026-06': 0,
}

FINAL_DF_CSV_CANDIDATES = [
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]

MONEY_METRICS = [
    'chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
    'commission_total', 'int_component', 'aur', 'amortization', 'trx_sum',
]
COUNT_METRICS = ['unique_inn', 'retl_cnt', 'term_cnt', 'trx_cnt', 'agr_rows', 'agr_with_chod_gt0']


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower:
            return lower[key]
    for c in cols:
        cl = str(c).strip().lower().replace('\n', ' ')
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce',
    )


print('Focus month:', FOCUS_MONTH)
print('Excel files:')
for m, p in excel_reference_by_month.items():
    print(f'  {m}: exists={p.exists()} header={excel_header_by_month.get(m, 0)} | {p.name}')

def clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in {'None', 'nan', 'NaN'}:
            out.append(s)
    return sorted(set(out))


def sql_in(values):
    vals = clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


## 1) Загрузка Excel → agr-level + monthly totals


In [ ]:
COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия эквайринга', 'Комиссия (% с операций)',
        'Комиссия \n(% с операций)', 'Комиссия % с операций',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    ],
    'comm_total_col': [
        'Общая комиссия', 'Комиссия общая', 'Итого комиссия',
        'Итоговая комиссия', 'Коммиссия эквайринга',
    ],
    'int_component_col': [
        'Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)',
        'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)',
    ],
    'chod_col': ['ЧОД'],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
}


def load_excel_month(report_month, excel_path, excel_header=0):
    ex = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in COL_MAP.items()}
    required = ['inn_col', 'agr_col', 'chod_col']
    missing = [k for k in required if resolved.get(k) is None]
    if missing:
        raise ValueError(
            f'{report_month}: missing {missing}. columns={list(ex.columns)}'
        )

    out = pd.DataFrame({
        'report_month': report_month,
        'inn_key': ex[resolved['inn_col']].map(normalize_inn_q1),
        'agr_id_key': ex[resolved['agr_col']].map(normalize_agr_q1),
    })

    def take(name, dest, default=np.nan):
        col = resolved.get(name)
        if col is None:
            out[dest] = default
        else:
            out[dest] = to_num_series(ex[col])

    take('retl_col', 'retl_cnt')
    take('term_col', 'term_cnt')
    take('trx_cnt_col', 'trx_cnt')
    take('trx_sum_col', 'trx_sum')
    take('comm_ops_col', 'commission_from_ops')
    take('comm_monthly_col', 'commission_monthly')
    take('comm_total_col', 'commission_total')
    take('int_component_col', 'int_component')
    take('chod_col', 'chod')
    take('aur_col', 'aur')
    take('amortization_col', 'amortization')
    take('fin_result_col', 'fin_result')

    # fill commission_total if absent
    if out['commission_total'].isna().all():
        out['commission_total'] = (
            out['commission_from_ops'].fillna(0) + out['commission_monthly'].fillna(0)
        )

    agr = (
        out.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt': 'max',
            'term_cnt': 'max',
            'trx_cnt': 'max',
            'trx_sum': 'sum',
            'commission_from_ops': 'sum',
            'commission_monthly': 'sum',
            'commission_total': 'sum',
            'int_component': 'sum',
            'chod': 'sum',
            'aur': 'sum',
            'amortization': 'sum',
            'fin_result': 'sum',
        })
    )
    return agr, resolved


excel_agr_parts = []
resolved_by_month = {}
for month, path in excel_reference_by_month.items():
    if not path.exists():
        print('SKIP missing:', month, path)
        continue
    header = int(excel_header_by_month.get(month, 0))
    agr_df, resolved = load_excel_month(month, path, excel_header=header)
    excel_agr_parts.append(agr_df)
    resolved_by_month[month] = resolved
    print(
        f'{month}: rows_agr={len(agr_df)} | chod={agr_df["chod"].fillna(0).sum():,.0f} '
        f'| fin={agr_df["fin_result"].fillna(0).sum():,.0f} | header={header}'
    )
    print('  cols:', {k: v for k, v in resolved.items() if v is not None})

if not excel_agr_parts:
    raise RuntimeError('No Excel months loaded')

excel_agr_df = pd.concat(excel_agr_parts, ignore_index=True)
print('excel_agr_df rows =', len(excel_agr_df))
display(excel_agr_df.head(3))


## 2) Помесячные тоталы Excel + просадка марта


In [ ]:
def month_totals(df):
    g = df.groupby('report_month', as_index=False).agg(
        unique_inn=('inn_key', 'nunique'),
        agr_rows=('agr_id_key', 'nunique'),
        agr_with_chod_gt0=('chod', lambda s: int((pd.to_numeric(s, errors='coerce').fillna(0) > 0).sum())),
        retl_cnt=('retl_cnt', 'sum'),
        term_cnt=('term_cnt', 'sum'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        commission_monthly=('commission_monthly', 'sum'),
        commission_total=('commission_total', 'sum'),
        int_component=('int_component', 'sum'),
        chod=('chod', 'sum'),
        aur=('aur', 'sum'),
        amortization=('amortization', 'sum'),
        fin_result=('fin_result', 'sum'),
    )
    # identity check components
    g['fin_result_recalc'] = g['chod'].fillna(0) - g['aur'].fillna(0) - g['amortization'].fillna(0)
    g['fin_identity_gap'] = g['fin_result'].fillna(0) - g['fin_result_recalc']
    g['chod_minus_comm_int'] = (
        g['chod'].fillna(0) - g['commission_total'].fillna(0) - g['int_component'].fillna(0)
    )
    return g.sort_values('report_month')


monthly_excel = month_totals(excel_agr_df)
print('=== Excel monthly totals ===')
display(monthly_excel)

# March vs peers
if FOCUS_MONTH not in set(monthly_excel['report_month']):
    raise RuntimeError(f'{FOCUS_MONTH} not in loaded months')

march = monthly_excel.loc[monthly_excel['report_month'] == FOCUS_MONTH].iloc[0]
peers = monthly_excel.loc[monthly_excel['report_month'].isin(PEER_MONTHS)].copy()
peer_mean = peers.select_dtypes(include=[np.number]).mean(numeric_only=True)

cmp_rows = []
for metric in ['chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
               'commission_total', 'int_component', 'aur', 'amortization',
               'trx_sum', 'trx_cnt', 'term_cnt', 'unique_inn', 'agr_rows']:
    m_val = float(march[metric]) if pd.notna(march[metric]) else np.nan
    p_val = float(peer_mean[metric]) if metric in peer_mean.index and pd.notna(peer_mean[metric]) else np.nan
    feb = monthly_excel.loc[monthly_excel['report_month'] == '2026-02', metric]
    apr = monthly_excel.loc[monthly_excel['report_month'] == '2026-04', metric]
    feb_v = float(feb.iloc[0]) if len(feb) else np.nan
    apr_v = float(apr.iloc[0]) if len(apr) else np.nan
    cmp_rows.append({
        'metric': metric,
        'march': m_val,
        'peer_mean_excl_march': p_val,
        'delta_vs_peer_mean': m_val - p_val if pd.notna(m_val) and pd.notna(p_val) else np.nan,
        'pct_vs_peer_mean': (m_val / p_val - 1.0) if pd.notna(m_val) and p_val not in (0, np.nan) else np.nan,
        'feb': feb_v,
        'apr': apr_v,
        'delta_march_minus_feb': m_val - feb_v if pd.notna(m_val) and pd.notna(feb_v) else np.nan,
        'delta_march_minus_apr': m_val - apr_v if pd.notna(m_val) and pd.notna(apr_v) else np.nan,
    })

march_vs_peers = pd.DataFrame(cmp_rows)
print('=== March vs peer months (Excel) ===')
display(march_vs_peers)

# Contribution to fin_result dip vs peer mean
print('=== Decomposition of March fin_result vs peer mean ===')
# fin = chod - aur - amort
for part in ['chod', 'aur', 'amortization', 'fin_result']:
    row = march_vs_peers.loc[march_vs_peers['metric'] == part].iloc[0]
    sign = '-' if part in {'aur', 'amortization'} else ''
    print(
        f'{sign}{part}: march={row["march"]:,.0f} | peer_mean={row["peer_mean_excl_march"]:,.0f} '
        f'| delta={row["delta_vs_peer_mean"]:,.0f} ({row["pct_vs_peer_mean"]*100 if pd.notna(row["pct_vs_peer_mean"]) else np.nan:.1f}%)'
    )

chod_d = float(march_vs_peers.loc[march_vs_peers['metric']=='chod', 'delta_vs_peer_mean'].iloc[0])
aur_d = float(march_vs_peers.loc[march_vs_peers['metric']=='aur', 'delta_vs_peer_mean'].iloc[0])
am_d = float(march_vs_peers.loc[march_vs_peers['metric']=='amortization', 'delta_vs_peer_mean'].iloc[0])
fin_d = float(march_vs_peers.loc[march_vs_peers['metric']=='fin_result', 'delta_vs_peer_mean'].iloc[0])
# d(fin) ≈ d(chod) - d(aur) - d(amort)
print(
    f'Check: d(chod)-d(aur)-d(amort) = {chod_d - aur_d - am_d:,.0f} '
    f'vs d(fin_result) = {fin_d:,.0f}'
)

id_gap = float(march['fin_identity_gap']) if 'fin_identity_gap' in march.index else np.nan
print(f'March Excel identity gap fin - (chod-aur-amort) = {id_gap:,.2f}')


## 3) Какие договоры дали просадку (March vs Feb / March vs Apr)

TOP agr по падению `chod` и `fin_result`.  
Также: договоры, которые были в Feb/Apr, но пропали в March (или наоборот).


In [ ]:
def pivot_metric(df, metric):
    p = (
        df.pivot_table(
            index=['inn_key', 'agr_id_key'],
            columns='report_month',
            values=metric,
            aggfunc='sum',
        )
        .reset_index()
    )
    return p


def build_pair_delta(piv, left_m, right_m, metric_name):
    """right - left (e.g. march - feb): negative = drop in March."""
    out = piv[['inn_key', 'agr_id_key']].copy()
    out[left_m] = piv[left_m] if left_m in piv.columns else np.nan
    out[right_m] = piv[right_m] if right_m in piv.columns else np.nan
    out[left_m] = pd.to_numeric(out[left_m], errors='coerce').fillna(0)
    out[right_m] = pd.to_numeric(out[right_m], errors='coerce').fillna(0)
    out['delta'] = out[right_m] - out[left_m]
    out['metric'] = metric_name
    out['pair'] = f'{right_m}_minus_{left_m}'
    return out


chod_piv = pivot_metric(excel_agr_df, 'chod')
fin_piv = pivot_metric(excel_agr_df, 'fin_result')
ops_piv = pivot_metric(excel_agr_df, 'commission_from_ops')
trx_piv = pivot_metric(excel_agr_df, 'trx_sum')

pairs = []
for metric_name, piv in [('chod', chod_piv), ('fin_result', fin_piv),
                         ('commission_from_ops', ops_piv), ('trx_sum', trx_piv)]:
    if '2026-02' in piv.columns and FOCUS_MONTH in piv.columns:
        pairs.append(build_pair_delta(piv, '2026-02', FOCUS_MONTH, metric_name))
    if '2026-04' in piv.columns and FOCUS_MONTH in piv.columns:
        pairs.append(build_pair_delta(piv, '2026-04', FOCUS_MONTH, metric_name))

pair_df = pd.concat(pairs, ignore_index=True)

print('=== TOP-20 agr: CHOD drop March vs Feb ===')
top_chod_feb = (
    pair_df.loc[(pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')]
    .sort_values('delta')
    .head(20)
)
display(top_chod_feb)

print('=== TOP-20 agr: CHOD drop March vs Apr ===')
top_chod_apr = (
    pair_df.loc[(pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-04')]
    .sort_values('delta')
    .head(20)
)
display(top_chod_apr)

print('=== TOP-20 agr: fin_result drop March vs Feb ===')
top_fin_feb = (
    pair_df.loc[(pair_df['metric'] == 'fin_result') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')]
    .sort_values('delta')
    .head(20)
)
display(top_fin_feb)

print('=== TOP-20 agr: fin_result drop March vs Apr ===')
top_fin_apr = (
    pair_df.loc[(pair_df['metric'] == 'fin_result') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-04')]
    .sort_values('delta')
    .head(20)
)
display(top_fin_apr)

# Coverage / appearance
def presence(month):
    return set(
        zip(
            excel_agr_df.loc[excel_agr_df['report_month'] == month, 'inn_key'],
            excel_agr_df.loc[excel_agr_df['report_month'] == month, 'agr_id_key'],
        )
    )

keys_feb = presence('2026-02') if '2026-02' in excel_agr_df['report_month'].values else set()
keys_mar = presence(FOCUS_MONTH)
keys_apr = presence('2026-04') if '2026-04' in excel_agr_df['report_month'].values else set()

only_feb_not_mar = keys_feb - keys_mar
only_mar_not_feb = keys_mar - keys_feb
only_apr_not_mar = keys_apr - keys_mar
only_mar_not_apr = keys_mar - keys_apr

print('=== Key coverage ===')
print(f'Feb keys={len(keys_feb)} | Mar={len(keys_mar)} | Apr={len(keys_apr)}')
print(f'in Feb not Mar: {len(only_feb_not_mar)} | in Mar not Feb: {len(only_mar_not_feb)}')
print(f'in Apr not Mar: {len(only_apr_not_mar)} | in Mar not Apr: {len(only_mar_not_apr)}')

# How much CHOD drop explained by TOP-N / disappeared keys
chod_m_f = pair_df.loc[
    (pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')
].copy()
total_drop = float(chod_m_f.loc[chod_m_f['delta'] < 0, 'delta'].sum())
top20_drop = float(chod_m_f.nsmallest(20, 'delta')['delta'].sum())
print(f'CHOD March-Feb: sum of negative deltas={total_drop:,.0f} | TOP20 share={top20_drop/total_drop if total_drop else np.nan:.1%}')

# Disappeared agrs contribution (were in Feb with chod, absent in Mar)
if only_feb_not_mar:
    _gone = pd.DataFrame(list(only_feb_not_mar), columns=['inn_key', 'agr_id_key'])
    feb_gone = excel_agr_df.loc[excel_agr_df['report_month'] == '2026-02'].merge(
        _gone, on=['inn_key', 'agr_id_key'], how='inner'
    )
    print(
        f'Agr in Feb not Mar: n={len(feb_gone)} | their Feb CHOD sum={feb_gone["chod"].fillna(0).sum():,.0f} '
        f'| Feb fin={feb_gone["fin_result"].fillna(0).sum():,.0f}'
    )
    display(
        feb_gone.sort_values('chod', ascending=False)
        [['inn_key', 'agr_id_key', 'chod', 'fin_result', 'commission_from_ops', 'trx_sum', 'term_cnt']]
        .head(15)
    )


## 4) Разложение просадки ЧОД на компоненты по agr (March vs Feb)

Для TOP просевших agr смотрим, что упало сильнее: `commission_from_ops`, `commission_monthly`, `int_component`, `trx_sum`.


In [ ]:
def wide_compare(months=('2026-02', FOCUS_MONTH)):
    sub = excel_agr_df.loc[excel_agr_df['report_month'].isin(months)].copy()
    metrics = [
        'chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
        'commission_total', 'int_component', 'aur', 'amortization', 'trx_sum', 'trx_cnt', 'term_cnt',
    ]
    pieces = []
    for met in metrics:
        p = sub.pivot_table(
            index=['inn_key', 'agr_id_key'],
            columns='report_month',
            values=met,
            aggfunc='sum',
        )
        for m in months:
            if m not in p.columns:
                p[m] = np.nan
        p = p.reset_index()
        p[f'{met}_left'] = pd.to_numeric(p[months[0]], errors='coerce').fillna(0)
        p[f'{met}_right'] = pd.to_numeric(p[months[1]], errors='coerce').fillna(0)
        p[f'd_{met}'] = p[f'{met}_right'] - p[f'{met}_left']
        pieces.append(p[['inn_key', 'agr_id_key', f'{met}_left', f'{met}_right', f'd_{met}']])
    out = pieces[0]
    for p in pieces[1:]:
        out = out.merge(p, on=['inn_key', 'agr_id_key'], how='outer')
    return out


comp_feb_mar = wide_compare(('2026-02', FOCUS_MONTH))
comp_top = comp_feb_mar.sort_values('d_chod').head(30)
print('=== TOP-30 agr by d_chod (March−Feb) with component deltas ===')
display(comp_top[[
    'inn_key', 'agr_id_key',
    'chod_left', 'chod_right', 'd_chod',
    'd_commission_from_ops', 'd_commission_monthly', 'd_int_component',
    'd_trx_sum', 'd_trx_cnt', 'd_aur', 'd_amortization', 'd_fin_result',
]])

# Aggregate: share of total CHOD drop explained by commission_from_ops vs other
neg = comp_feb_mar.loc[comp_feb_mar['d_chod'] < 0].copy()
print('Among agr with CHOD drop March vs Feb:')
print(f'  sum d_chod = {neg["d_chod"].sum():,.0f}')
print(f'  sum d_commission_from_ops = {neg["d_commission_from_ops"].sum():,.0f}')
print(f'  sum d_commission_monthly = {neg["d_commission_monthly"].sum():,.0f}')
print(f'  sum d_int_component = {neg["d_int_component"].sum():,.0f}')
print(f'  sum d_trx_sum = {neg["d_trx_sum"].sum():,.0f}')
print(f'  sum d_fin_result = {neg["d_fin_result"].sum():,.0f}')


## 5) (Опционально) Excel March vs lake `final_df` — тот же месяц


In [ ]:
fdf = None
fdf_src = None
for pth in FINAL_DF_CSV_CANDIDATES:
    if pth.exists():
        tmp = pd.read_csv(pth, dtype=str, low_memory=False)
        tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
        fdf = tmp.loc[tmp['report_month'] == FOCUS_MONTH].copy()
        if len(fdf):
            fdf_src = str(pth)
            break

if fdf is None or not len(fdf):
    print('SKIP lake compare: final_df CSV for March not found')
else:
    print('Lake source:', fdf_src, 'rows=', len(fdf))
    fdf['inn_key'] = fdf['inn'].map(normalize_inn_q1)
    fdf['agr_id_key'] = fdf['agr_id'].map(normalize_agr_q1)
    for c in ['chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
              'aur', 'amortization', 'trx_sum', 'term_cnt']:
        if c in fdf.columns:
            fdf[c] = pd.to_numeric(fdf[c], errors='coerce')
        else:
            fdf[c] = np.nan

    lake_m = fdf.dropna(subset=['agr_id_key']).groupby(['inn_key', 'agr_id_key'], as_index=False).agg({
        'chod': 'max', 'fin_result': 'max', 'commission_from_ops': 'max',
        'commission_monthly': 'max', 'aur': 'max', 'amortization': 'max',
        'trx_sum': 'max', 'term_cnt': 'max',
    })
    ex_m = excel_agr_df.loc[excel_agr_df['report_month'] == FOCUS_MONTH].copy()
    both = ex_m.merge(lake_m, on=['inn_key', 'agr_id_key'], how='outer', suffixes=('_excel', '_lake'), indicator=True)
    for c in ['chod', 'fin_result', 'commission_from_ops', 'trx_sum']:
        both[f'd_{c}'] = both[f'{c}_lake'].fillna(0) - both[f'{c}_excel'].fillna(0)

    print('March totals Excel vs lake:')
    for c in ['chod', 'fin_result', 'commission_from_ops', 'aur', 'amortization']:
        e = float(pd.to_numeric(ex_m[c], errors='coerce').fillna(0).sum())
        l = float(pd.to_numeric(lake_m[c], errors='coerce').fillna(0).sum()) if c in lake_m.columns else np.nan
        print(f'  {c}: excel={e:,.0f} | lake={l:,.0f} | delta(lake-excel)={l-e:,.0f}')

    print('TOP-15 |d_chod| Excel vs lake (March):')
    display(
        both.assign(abs_d=both['d_chod'].abs())
        .sort_values('abs_d', ascending=False)
        [['inn_key', 'agr_id_key', 'chod_excel', 'chod_lake', 'd_chod',
          'fin_result_excel', 'fin_result_lake', 'd_fin_result', '_merge']]
        .head(15)
    )


## 5b) `final_df`: ЧОД по месяцам + расхождение AUR (lake vs Excel)

Сравниваем помесячно:
- `chod` Excel vs `chod` из `final_df`
- `aur` Excel vs `aur` из `final_df` (`retl_cnt * 1926` в озере)

Нужен CSV периода (`final_df_period_…_mpos.csv`). Impala не требуется.


In [ ]:
# 5b) final_df monthly CHOD + AUR lake vs Excel
from IPython.display import display

AUR_RATE = 1926.0

fdf_period = None
fdf_period_src = None
for pth in FINAL_DF_CSV_CANDIDATES:
    if not pth.exists():
        continue
    tmp = pd.read_csv(pth, dtype=str, low_memory=False)
    if 'report_month' not in tmp.columns:
        print('SKIP no report_month in', pth)
        continue
    tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
    fdf_period = tmp
    fdf_period_src = str(pth)
    break

if fdf_period is None or not len(fdf_period):
    raise RuntimeError(
        'final_df period CSV not found. Expected one of:\n'
        + '\n'.join(str(p) for p in FINAL_DF_CSV_CANDIDATES)
    )

print('final_df source:', fdf_period_src, '| rows=', len(fdf_period))
print('months:', sorted(fdf_period['report_month'].dropna().unique().tolist()))

fdf_period = fdf_period.copy()
fdf_period['inn_key'] = fdf_period['inn'].map(normalize_inn_q1) if 'inn' in fdf_period.columns else None
fdf_period['agr_id_key'] = fdf_period['agr_id'].map(normalize_agr_q1) if 'agr_id' in fdf_period.columns else None

num_cols = [
    'chod', 'fin_result', 'aur', 'amortization', 'retl_cnt', 'term_cnt',
    'trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly',
    'commission_total', 'int_component',
]
for c in num_cols:
    if c in fdf_period.columns:
        fdf_period[c] = pd.to_numeric(fdf_period[c], errors='coerce')
    else:
        fdf_period[c] = np.nan

# agr-level collapse (max), then month sum — same spirit as build_lake_agg
lake_agr = (
    fdf_period.dropna(subset=['agr_id_key', 'report_month'])
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg({
        'chod': 'max',
        'fin_result': 'max',
        'aur': 'max',
        'amortization': 'max',
        'retl_cnt': 'max',
        'term_cnt': 'max',
        'trx_cnt': 'max',
        'trx_sum': 'max',
        'commission_from_ops': 'max',
        'commission_monthly': 'max',
        'commission_total': 'max',
        'int_component': 'max',
    })
)

monthly_lake = (
    lake_agr.groupby('report_month', as_index=False)
    .agg(
        unique_inn=('inn_key', 'nunique'),
        agr_rows=('agr_id_key', 'nunique'),
        retl_cnt=('retl_cnt', 'sum'),
        term_cnt=('term_cnt', 'sum'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        commission_monthly=('commission_monthly', 'sum'),
        commission_total=('commission_total', 'sum'),
        int_component=('int_component', 'sum'),
        chod=('chod', 'sum'),
        aur=('aur', 'sum'),
        amortization=('amortization', 'sum'),
        fin_result=('fin_result', 'sum'),
    )
    .sort_values('report_month')
)
monthly_lake['aur_from_retl'] = monthly_lake['retl_cnt'].fillna(0) * AUR_RATE
monthly_lake['aur_vs_retl_x_rate'] = monthly_lake['aur'].fillna(0) - monthly_lake['aur_from_retl']

print('=== final_df monthly totals (lake) ===')
display(monthly_lake)

# Align with Excel monthly (already in monthly_excel from section 2)
ex = monthly_excel.copy()
for c in ['chod', 'fin_result', 'aur', 'amortization', 'retl_cnt', 'term_cnt',
          'trx_sum', 'commission_from_ops', 'commission_monthly']:
    if c not in ex.columns:
        ex[c] = np.nan

cmp = ex.merge(
    monthly_lake,
    on='report_month',
    how='outer',
    suffixes=('_excel', '_lake'),
).sort_values('report_month')

for metric in ['chod', 'aur', 'fin_result', 'retl_cnt', 'amortization', 'commission_from_ops']:
    ce, cl = f'{metric}_excel', f'{metric}_lake'
    if ce in cmp.columns and cl in cmp.columns:
        cmp[f'delta_{metric}_lake_minus_excel'] = (
            pd.to_numeric(cmp[cl], errors='coerce').fillna(0)
            - pd.to_numeric(cmp[ce], errors='coerce').fillna(0)
        )
        cmp[f'pct_{metric}_vs_excel'] = np.where(
            pd.to_numeric(cmp[ce], errors='coerce').fillna(0).abs() > 1e-9,
            cmp[f'delta_{metric}_lake_minus_excel'] / pd.to_numeric(cmp[ce], errors='coerce'),
            np.nan,
        )

print('=== CHOD by month: Excel vs final_df ===')
chod_cmp = cmp[[
    'report_month', 'chod_excel', 'chod_lake',
    'delta_chod_lake_minus_excel', 'pct_chod_vs_excel',
]].copy()
display(chod_cmp)

print('=== AUR by month: Excel vs final_df ===')
aur_cmp = cmp[[
    'report_month',
    'aur_excel', 'aur_lake',
    'retl_cnt_excel', 'retl_cnt_lake',
    'delta_aur_lake_minus_excel', 'pct_aur_vs_excel',
]].copy()
# lake identity: aur should ≈ retl*1926
if 'aur_from_retl' in monthly_lake.columns:
    aur_cmp = aur_cmp.merge(
        monthly_lake[['report_month', 'aur_from_retl', 'aur_vs_retl_x_rate']],
        on='report_month',
        how='left',
    )
aur_cmp['aur_excel_per_retl'] = np.where(
    pd.to_numeric(aur_cmp['retl_cnt_excel'], errors='coerce').fillna(0) > 0,
    pd.to_numeric(aur_cmp['aur_excel'], errors='coerce') / pd.to_numeric(aur_cmp['retl_cnt_excel'], errors='coerce'),
    np.nan,
)
aur_cmp['aur_lake_per_retl'] = np.where(
    pd.to_numeric(aur_cmp['retl_cnt_lake'], errors='coerce').fillna(0) > 0,
    pd.to_numeric(aur_cmp['aur_lake'], errors='coerce') / pd.to_numeric(aur_cmp['retl_cnt_lake'], errors='coerce'),
    np.nan,
)
display(aur_cmp)

print('=== fin_result by month: Excel vs final_df (context) ===')
display(cmp[[
    'report_month', 'fin_result_excel', 'fin_result_lake',
    'delta_fin_result_lake_minus_excel', 'pct_fin_result_vs_excel',
]])

# Is there a March CHOD dip in lake?
if FOCUS_MONTH in set(monthly_lake['report_month']):
    lk = monthly_lake.set_index('report_month')
    m_chod = float(lk.loc[FOCUS_MONTH, 'chod'])
    peer = [m for m in PEER_MONTHS if m in lk.index]
    peer_mean_chod = float(lk.loc[peer, 'chod'].mean()) if peer else np.nan
    print('=== Lake CHOD March check ===')
    print(f'March lake CHOD = {m_chod:,.2f}')
    print(f'Peer mean lake CHOD = {peer_mean_chod:,.2f}')
    if peer_mean_chod:
        print(f'March / peer_mean = {m_chod/peer_mean_chod:.3f} ({(m_chod/peer_mean_chod-1)*100:.1f}%)')
    print('Lake CHOD by month:')
    for m, v in lk['chod'].items():
        mark = ' <-- focus' if m == FOCUS_MONTH else ''
        print(f'  {m}: {float(v):,.2f}{mark}')

# AUR summary verdict
print('=== AUR verdict ===')
for _, r in aur_cmp.iterrows():
    m = r['report_month']
    d = r.get('delta_aur_lake_minus_excel')
    pct = r.get('pct_aur_vs_excel')
    per_e = r.get('aur_excel_per_retl')
    per_l = r.get('aur_lake_per_retl')
    print(
        f'{m}: delta(lake-excel)={d:,.0f} ({pct*100 if pd.notna(pct) else float("nan"):+.1f}%) | '
        f'AUR/retl excel={per_e:,.1f} lake={per_l:,.1f} (expect lake≈{AUR_RATE:.0f})'
    )

# Save
out_lake_monthly = OUTPUT_DIR / 'final_df_monthly_chod_aur_2026.csv'
out_cmp = OUTPUT_DIR / 'excel_vs_final_df_chod_aur_by_month.csv'
monthly_lake.to_csv(out_lake_monthly, index=False, encoding='utf-8-sig')
cmp.to_csv(out_cmp, index=False, encoding='utf-8-sig')
chod_cmp.to_csv(OUTPUT_DIR / 'excel_vs_final_df_chod_by_month.csv', index=False, encoding='utf-8-sig')
aur_cmp.to_csv(OUTPUT_DIR / 'excel_vs_final_df_aur_by_month.csv', index=False, encoding='utf-8-sig')
print('Saved:', out_lake_monthly)
print('Saved:', out_cmp)


## 5c) Примеры AUR в Excel за апрель, где ставка ≠ 1926

Для ручной проверки в отчёте `04_Апрель_2026.xlsx`.

На зерне `inn + agr_id`:
- `aur_per_retl = АУР / Кол-во торговых точек`
- отбираем строки, где `|aur_per_retl - 1926| > 1` (и retl > 0)

Показываем TOP по `|АУР|` и распределение ставок.


In [ ]:
# 5c) April Excel AUR examples where rate != 1926
from IPython.display import display

AUR_RATE = 1926.0
APRIL = '2026-04'
TOL = 1.0  # rub per point

if 'excel_agr_df' not in globals() or excel_agr_df is None or not len(excel_agr_df):
    raise RuntimeError('Сначала выполни секцию 1 (excel_agr_df)')

apr = excel_agr_df.loc[excel_agr_df['report_month'] == APRIL].copy()
if not len(apr):
    raise RuntimeError(f'No Excel rows for {APRIL}')

apr['aur'] = pd.to_numeric(apr['aur'], errors='coerce')
apr['retl_cnt'] = pd.to_numeric(apr['retl_cnt'], errors='coerce')
apr['aur_per_retl'] = np.where(
    apr['retl_cnt'].fillna(0) > 0,
    apr['aur'] / apr['retl_cnt'],
    np.nan,
)
apr['delta_rate_vs_1926'] = apr['aur_per_retl'] - AUR_RATE
apr['abs_delta_rate'] = apr['delta_rate_vs_1926'].abs()

has_retl = apr['retl_cnt'].fillna(0) > 0
rate_ne = has_retl & apr['aur_per_retl'].notna() & (apr['abs_delta_rate'] > TOL)
rate_eq = has_retl & apr['aur_per_retl'].notna() & (apr['abs_delta_rate'] <= TOL)
no_retl = ~has_retl

print(f'=== Excel {APRIL} AUR rate vs {AUR_RATE:.0f} ===')
print(f'agr rows: {len(apr)}')
print(f'  retl>0 & rate≈1926: {int(rate_eq.sum())}')
print(f'  retl>0 & rate≠1926: {int(rate_ne.sum())}')
print(f'  retl=0/empty: {int(no_retl.sum())}')
print(
    f'  AUR sum where rate≠1926: {apr.loc[rate_ne, "aur"].fillna(0).sum():,.2f} '
    f'({apr.loc[rate_ne, "aur"].fillna(0).sum() / max(apr["aur"].fillna(0).sum(), 1):.1%} of April Excel AUR)'
)

# rate distribution (rounded)
apr_pos = apr.loc[has_retl & apr['aur_per_retl'].notna()].copy()
apr_pos['rate_round'] = apr_pos['aur_per_retl'].round(2)
rate_dist = (
    apr_pos.groupby('rate_round', as_index=False)
    .agg(agr_cnt=('agr_id_key', 'nunique'), aur_sum=('aur', 'sum'), retl_sum=('retl_cnt', 'sum'))
    .sort_values('aur_sum', ascending=False)
)
print('=== Distinct AUR/retl rates in April Excel (by AUR sum) ===')
display(rate_dist.head(20))

examples = (
    apr.loc[rate_ne]
    .sort_values('aur', ascending=False, key=lambda s: s.abs())
    [['inn_key', 'agr_id_key', 'retl_cnt', 'aur', 'aur_per_retl', 'delta_rate_vs_1926',
      'term_cnt', 'chod', 'fin_result']]
    .head(40)
)
print(f'=== TOP-40 agr in April Excel with |AUR/retl - {AUR_RATE:.0f}| > {TOL} ===')
print('Проверь эти ИНН/договоры глазами в 04_Апрель_2026.xlsx (колонки АУР и Кол-во торговых точек).')
display(examples)

# Also show a few with rate exactly 1926 for contrast
eq_examples = (
    apr.loc[rate_eq]
    .sort_values('aur', ascending=False)
    [['inn_key', 'agr_id_key', 'retl_cnt', 'aur', 'aur_per_retl', 'term_cnt']]
    .head(10)
)
print('=== Для сравнения: 10 agr с rate≈1926 ===')
display(eq_examples)

out_ex = OUTPUT_DIR / 'excel_april_2026_aur_rate_ne_1926_examples.csv'
out_dist = OUTPUT_DIR / 'excel_april_2026_aur_rate_distribution.csv'
examples.to_csv(out_ex, index=False, encoding='utf-8-sig')
rate_dist.to_csv(out_dist, index=False, encoding='utf-8-sig')
print('Saved:', out_ex)
print('Saved:', out_dist)


## 5d) Причина просадки CHOD в марте в Озере (`final_df`)

Excel в марте нормальный; проседает **только lake**.

Разбор:
1. Помесячные компоненты lake: `commission_from_ops`, `commission_monthly`, `int_component`, `chod`.
2. March vs Feb / March vs Apr — что упало в тоталах.
3. TOP agr в lake по падению CHOD (Mar−Feb).
4. Те же agr: Excel March vs lake March (есть ли дыра только в озере).


In [ ]:
# 5d) March lake CHOD dip root-cause
from IPython.display import display

if 'lake_agr' not in globals() or lake_agr is None or not len(lake_agr):
    raise RuntimeError('Сначала выполни 5b (нужен lake_agr из final_df)')
if 'excel_agr_df' not in globals() or excel_agr_df is None or not len(excel_agr_df):
    raise RuntimeError('Сначала выполни секцию 1 (excel_agr_df)')

COMP_COLS = [
    'commission_from_ops', 'commission_monthly', 'commission_total',
    'int_component', 'chod', 'trx_sum', 'trx_cnt', 'term_cnt', 'retl_cnt',
    'aur', 'amortization', 'fin_result',
]

# --- 1) Lake monthly component view ---
lake_comp = (
    lake_agr.groupby('report_month', as_index=False)
    .agg({c: 'sum' for c in COMP_COLS if c in lake_agr.columns})
    .sort_values('report_month')
)
# identity: chod ≈ commission_total + int_component (lake formula)
lake_comp['chod_recalc'] = (
    lake_comp.get('commission_total', 0).fillna(0)
    + lake_comp.get('int_component', 0).fillna(0)
)
lake_comp['chod_identity_gap'] = lake_comp['chod'].fillna(0) - lake_comp['chod_recalc']

print('=== Lake monthly components (CHOD drivers) ===')
display(lake_comp)

focus_months = ['2026-02', FOCUS_MONTH, '2026-04']
sub = lake_comp.loc[lake_comp['report_month'].isin(focus_months)].set_index('report_month')
print('=== Lake totals Feb / Mar / Apr ===')
display(sub)

if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    print('=== March − Feb (lake totals) ===')
    d = sub.loc[FOCUS_MONTH] - sub.loc['2026-02']
    for c in COMP_COLS + ['chod_recalc']:
        if c in d.index:
            print(f'  d_{c}: {float(d[c]):,.2f}')

if FOCUS_MONTH in sub.index and '2026-04' in sub.index:
    print('=== March − Apr (lake totals) ===')
    d = sub.loc[FOCUS_MONTH] - sub.loc['2026-04']
    for c in ['chod', 'commission_from_ops', 'commission_monthly', 'int_component', 'trx_sum', 'fin_result']:
        if c in d.index:
            print(f'  d_{c}: {float(d[c]):,.2f}')

# Which component explains CHOD drop vs Feb?
if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    d_chod = float(sub.loc[FOCUS_MONTH, 'chod'] - sub.loc['2026-02', 'chod'])
    d_ops = float(sub.loc[FOCUS_MONTH, 'commission_from_ops'] - sub.loc['2026-02', 'commission_from_ops'])
    d_mon = float(sub.loc[FOCUS_MONTH, 'commission_monthly'] - sub.loc['2026-02', 'commission_monthly'])
    d_int = float(sub.loc[FOCUS_MONTH, 'int_component'] - sub.loc['2026-02', 'int_component'])
    d_tot = float(sub.loc[FOCUS_MONTH, 'commission_total'] - sub.loc['2026-02', 'commission_total'])
    print('=== Driver share of lake CHOD drop (Mar−Feb) ===')
    print(f'd_chod={d_chod:,.0f}')
    print(f'd_commission_total={d_tot:,.0f} (ops={d_ops:,.0f} + monthly={d_mon:,.0f})')
    print(f'd_int_component={d_int:,.0f}')
    print(f'd_ops+d_mon+d_int={d_ops+d_mon+d_int:,.0f} (should ≈ d_chod if formula holds)')


# --- 2) Agr-level lake: Mar vs Feb ---
def lake_pivot(metric):
    return lake_agr.pivot_table(
        index=['inn_key', 'agr_id_key'],
        columns='report_month',
        values=metric,
        aggfunc='max',
    ).reset_index()


chod_l = lake_pivot('chod')
ops_l = lake_pivot('commission_from_ops')
int_l = lake_pivot('int_component')
trx_l = lake_pivot('trx_sum')

def pair_delta(piv, left, right, name):
    out = piv[['inn_key', 'agr_id_key']].copy()
    out[left] = pd.to_numeric(piv[left], errors='coerce').fillna(0) if left in piv.columns else 0.0
    out[right] = pd.to_numeric(piv[right], errors='coerce').fillna(0) if right in piv.columns else 0.0
    out['delta'] = out[right] - out[left]
    out['metric'] = name
    return out

lake_chod_mf = pair_delta(chod_l, '2026-02', FOCUS_MONTH, 'chod')
lake_ops_mf = pair_delta(ops_l, '2026-02', FOCUS_MONTH, 'commission_from_ops')
lake_int_mf = pair_delta(int_l, '2026-02', FOCUS_MONTH, 'int_component')
lake_trx_mf = pair_delta(trx_l, '2026-02', FOCUS_MONTH, 'trx_sum')

def _rename_pair(df, left, right, dname, lname, rname):
    return df.rename(columns={left: lname, right: rname, 'delta': dname})[
        ['inn_key', 'agr_id_key', lname, rname, dname]
    ]


top_lake_drop = (
    _rename_pair(lake_chod_mf, '2026-02', FOCUS_MONTH, 'd_chod', 'chod_feb', 'chod_mar')
    .sort_values('d_chod')
    .head(30)
    .merge(_rename_pair(lake_ops_mf, '2026-02', FOCUS_MONTH, 'd_ops', 'ops_feb', 'ops_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
    .merge(_rename_pair(lake_int_mf, '2026-02', FOCUS_MONTH, 'd_int', 'int_feb', 'int_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
    .merge(_rename_pair(lake_trx_mf, '2026-02', FOCUS_MONTH, 'd_trx_sum', 'trx_feb', 'trx_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
)

print('=== TOP-30 agr lake CHOD drop March−Feb ===')
display(top_lake_drop[[
    'inn_key', 'agr_id_key', 'chod_feb', 'chod_mar', 'd_chod',
    'ops_feb', 'ops_mar', 'd_ops',
    'int_feb', 'int_mar', 'd_int',
    'trx_feb', 'trx_mar', 'd_trx_sum',
]])

neg = lake_chod_mf.loc[lake_chod_mf['delta'] < 0]
total_neg = float(neg['delta'].sum())
top30_neg = float(lake_chod_mf.nsmallest(30, 'delta')['delta'].sum())
print(f'sum negative d_chod (all agr)={total_neg:,.0f} | TOP30 share={top30_neg/total_neg if total_neg else np.nan:.1%}')


# --- 3) For TOP lake droppers: what does Excel say in March? ---
ex_mar = excel_agr_df.loc[excel_agr_df['report_month'] == FOCUS_MONTH][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum', 'fin_result']
].copy()
ex_mar = ex_mar.rename(columns={
    'chod': 'chod_excel',
    'commission_from_ops': 'ops_excel',
    'int_component': 'int_excel',
    'trx_sum': 'trx_excel',
    'fin_result': 'fin_excel',
})
lk_mar = lake_agr.loc[lake_agr['report_month'] == FOCUS_MONTH][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum', 'fin_result']
].copy()
lk_mar = lk_mar.rename(columns={
    'chod': 'chod_lake',
    'commission_from_ops': 'ops_lake',
    'int_component': 'int_lake',
    'trx_sum': 'trx_lake',
    'fin_result': 'fin_lake',
})

top_vs_excel = (
    top_lake_drop[['inn_key', 'agr_id_key', 'chod_feb', 'chod_mar', 'd_chod']]
    .rename(columns={'chod_feb': 'chod_lake_feb', 'chod_mar': 'chod_lake_mar', 'd_chod': 'd_chod_lake_mar_feb'})
    .merge(ex_mar, on=['inn_key', 'agr_id_key'], how='left')
    .merge(lk_mar, on=['inn_key', 'agr_id_key'], how='left')
)
top_vs_excel['d_chod_lake_minus_excel_mar'] = (
    top_vs_excel['chod_lake'].fillna(0) - top_vs_excel['chod_excel'].fillna(0)
)
top_vs_excel['d_ops_lake_minus_excel_mar'] = (
    top_vs_excel['ops_lake'].fillna(0) - top_vs_excel['ops_excel'].fillna(0)
)

print('=== TOP lake CHOD droppers: Excel March vs lake March ===')
print('Если chod_excel ≈ chod_lake_feb, а chod_lake_mar просел — дыра в озёрном March.')
display(top_vs_excel[[
    'inn_key', 'agr_id_key',
    'chod_lake_feb', 'chod_lake_mar', 'd_chod_lake_mar_feb',
    'chod_excel', 'd_chod_lake_minus_excel_mar',
    'ops_excel', 'ops_lake', 'd_ops_lake_minus_excel_mar',
    'trx_excel', 'trx_lake',
]])

# Aggregate March lake vs excel among all agr
both_mar = ex_mar.merge(lk_mar, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
both_mar['d_chod'] = both_mar['chod_lake'].fillna(0) - both_mar['chod_excel'].fillna(0)
both_mar['d_ops'] = both_mar['ops_lake'].fillna(0) - both_mar['ops_excel'].fillna(0)
both_mar['d_int'] = both_mar['int_lake'].fillna(0) - both_mar['int_excel'].fillna(0)
both_mar['d_trx'] = both_mar['trx_lake'].fillna(0) - both_mar['trx_excel'].fillna(0)

print('=== March coverage Excel vs lake ===')
print(both_mar['_merge'].value_counts(dropna=False).to_string())
print(
    f'March sum d_chod (lake-excel) all agr={both_mar["d_chod"].sum():,.0f} | '
    f'sum d_ops={both_mar["d_ops"].sum():,.0f} | sum d_int={both_mar["d_int"].sum():,.0f} | '
    f'sum d_trx={both_mar["d_trx"].sum():,.0f}'
)

top_gap = both_mar.sort_values('d_chod').head(30)
print('=== TOP-30 agr by lake−excel CHOD in March (most negative) ===')
display(top_gap[[
    'inn_key', 'agr_id_key', 'chod_excel', 'chod_lake', 'd_chod',
    'ops_excel', 'ops_lake', 'd_ops', 'int_excel', 'int_lake', 'd_int',
    'trx_excel', 'trx_lake', 'd_trx', '_merge',
]])

# Verdict
print('=== VERDICT (March lake CHOD) ===')
if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    # classify primary driver
    parts = {
        'commission_from_ops': d_ops,
        'commission_monthly': d_mon,
        'int_component': d_int,
    }
    # for CHOD drop, more negative = stronger driver
    primary = min(parts.items(), key=lambda kv: kv[1])
    print(f'Primary total-level driver Mar−Feb: {primary[0]} = {primary[1]:,.0f}')
    if abs(d_ops) >= abs(d_chod) * 0.5:
        print('→ Смотри trx / commission_from_ops пайплайн за март (section trx/ops).')
    elif abs(d_int) >= abs(d_chod) * 0.5:
        print('→ Смотри int_component / IRF за март.')
    else:
        print('→ Смешанный эффект; смотри TOP agr и d_ops/d_int на них.')

out_top = OUTPUT_DIR / 'lake_march_2026_top_chod_drop_vs_feb.csv'
out_gap = OUTPUT_DIR / 'lake_vs_excel_march_2026_top_chod_gap.csv'
out_comp = OUTPUT_DIR / 'lake_monthly_chod_components_2026.csv'
top_lake_drop.to_csv(out_top, index=False, encoding='utf-8-sig')
top_gap.to_csv(out_gap, index=False, encoding='utf-8-sig')
lake_comp.to_csv(out_comp, index=False, encoding='utf-8-sig')
print('Saved:', out_top)
print('Saved:', out_gap)
print('Saved:', out_comp)


## 5e) Deep-dive: почему в марте в Озере «поплыл» IRF (`int_component`)

Гипотезы (проверяем):
1. В марте в `scd1_trx_int` **больше/другие** `n_amt_fee` на том же периметре trx.
2. **Дубли** строк `trx_int` на один `n_trx` → `sum(fee)` раздувается.
3. Сдвиг **знака** fee (pos/neg mix) только в марте.
4. Расхождение lake vs Excel по IRF на TOP agr (при живых ops/trx).
5. Пересчёт IRF из Impala за Feb/Mar/Apr тем же периметром, что витрина → совпадает ли с `final_df`.

Нужен Impala + уже посчитанные `excel_agr_df` / `lake_agr` (секции 1 и 5b).


In [ ]:
# 5e.0) Impala connect (если ещё нет)
from rail_connectors.connection import connect

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp connection')


In [ ]:
# 5e) March IRF / trx_int diagnostics
from IPython.display import display
from calendar import monthrange

if 'excel_agr_df' not in globals() or excel_agr_df is None or not len(excel_agr_df):
    raise RuntimeError('Нужна секция 1 (excel_agr_df)')
if 'lake_agr' not in globals() or lake_agr is None or not len(lake_agr):
    raise RuntimeError('Нужна секция 5b (lake_agr)')

COMPARE_MONTHS = ['2026-02', '2026-03', '2026-04']


def month_bounds(ym: str):
    y, m = map(int, ym.split('-'))
    start = f'{ym}-01'
    end = f'{ym}-{monthrange(y, m)[1]:02d}'
    return start, end


# ---------- A) final_df vs Excel: int / ops / chod by month ----------
def _month_sum(df, month, col):
    sub = df.loc[df['report_month'] == month]
    return float(pd.to_numeric(sub[col], errors='coerce').fillna(0).sum())


rows = []
for ym in COMPARE_MONTHS + ['2026-01', '2026-05', '2026-06']:
    if ym not in set(excel_agr_df['report_month']) and ym not in set(lake_agr['report_month']):
        continue
    row = {'report_month': ym}
    for src_name, df in [('excel', excel_agr_df), ('lake', lake_agr)]:
        for col in ['chod', 'commission_from_ops', 'int_component', 'trx_sum', 'trx_cnt']:
            if col in df.columns and ym in set(df['report_month']):
                row[f'{col}_{src_name}'] = _month_sum(df, ym, col)
            else:
                row[f'{col}_{src_name}'] = np.nan
    row['d_int_lake_minus_excel'] = row.get('int_component_lake', np.nan) - row.get('int_component_excel', np.nan)
    row['d_ops_lake_minus_excel'] = row.get('commission_from_ops_lake', np.nan) - row.get('commission_from_ops_excel', np.nan)
    row['d_chod_lake_minus_excel'] = row.get('chod_lake', np.nan) - row.get('chod_excel', np.nan)
    rows.append(row)

cmp_irf = pd.DataFrame(rows).sort_values('report_month')
print('=== int_component / ops / chod: lake vs Excel ===')
display(cmp_irf)
print('Если только март аномален по d_int — проблема в данных/периметре марта, не в формуле.')


# ---------- B) Impala: same mart perimeter IRF metrics Feb/Mar/Apr ----------
def sql_irf_month(month_start: str, month_end: str) -> str:
    return f"""
    with fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base_raw as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{month_end}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
    ),
    trx_base as (
      select n_trx from trx_base_raw group by n_trx
    ),
    ta as (
      select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr,
             max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string), cast(a.n_agr as string)
    ),
    trx_keys as (
      select distinct n_trx from ta
    ),
    int_rows as (
      select
        cast(i.n_trx as string) as n_trx,
        coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee,
        coalesce(i.ods_deleted_flg, '0') as ods_deleted_flg
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
    ),
    int_agg as (
      select
        n_trx,
        sum(n_amt_fee) as fee_sum,
        sum(case when coalesce(ods_deleted_flg, '0') <> '1' then n_amt_fee else 0 end) as fee_sum_alive,
        count(*) as int_row_cnt,
        sum(case when coalesce(ods_deleted_flg, '0') <> '1' then 1 else 0 end) as int_row_alive_cnt,
        sum(case when n_amt_fee > 0 then 1 else 0 end) as fee_pos_rows,
        sum(case when n_amt_fee < 0 then 1 else 0 end) as fee_neg_rows,
        sum(case when n_amt_fee = 0 then 1 else 0 end) as fee_zero_rows
      from int_rows
      group by n_trx
    )
    select
      count(distinct t.n_trx) as trx_with_acq_sa,
      sum(t.n_amt_tax) as commission_from_ops,
      sum(coalesce(ia.fee_sum, 0.0)) as int_component_sum_all_rows,
      sum(coalesce(ia.fee_sum_alive, 0.0)) as int_component_sum_alive_only,
      sum(case when ia.n_trx is not null then 1 else 0 end) as trx_with_any_int,
      sum(case when coalesce(ia.int_row_cnt, 0) > 1 then 1 else 0 end) as trx_with_multi_int_rows,
      sum(case when coalesce(ia.int_row_alive_cnt, 0) > 1 then 1 else 0 end) as trx_with_multi_alive_int,
      sum(coalesce(ia.int_row_cnt, 0)) as int_rows_total,
      sum(coalesce(ia.fee_pos_rows, 0)) as fee_pos_rows,
      sum(coalesce(ia.fee_neg_rows, 0)) as fee_neg_rows,
      sum(coalesce(ia.fee_zero_rows, 0)) as fee_zero_rows,
      sum(case when coalesce(ia.fee_sum, 0.0) > 0 then 1 else 0 end) as trx_fee_sum_pos,
      sum(case when coalesce(ia.fee_sum, 0.0) < 0 then 1 else 0 end) as trx_fee_sum_neg,
      avg(case when ia.n_trx is not null then ia.fee_sum end) as avg_fee_per_trx_with_int,
      avg(case when ia.n_trx is not null then ia.int_row_cnt end) as avg_int_rows_per_trx
    from (
      select n_trx, sum(n_amt_tax) as n_amt_tax
      from ta
      group by n_trx
    ) t
    left join int_agg ia on ia.n_trx = t.n_trx
    """


impala_irf_parts = []
with imp:
    imp.execute('set MEM_LIMIT=16g')
    for ym in COMPARE_MONTHS:
        ms, me = month_bounds(ym)
        print(f'Impala IRF perimeter {ym}: {ms} .. {me} ...')
        part = imp.fetch(sql_irf_month(ms, me))
        if part is None or not len(part):
            print('  EMPTY')
            continue
        part = part.copy()
        part.insert(0, 'report_month', ym)
        impala_irf_parts.append(part)
        print(
            f'  ops={float(part.iloc[0]["commission_from_ops"]):,.0f} | '
            f'int_all={float(part.iloc[0]["int_component_sum_all_rows"]):,.0f} | '
            f'int_alive={float(part.iloc[0]["int_component_sum_alive_only"]):,.0f} | '
            f'multi_int_trx={int(part.iloc[0]["trx_with_multi_int_rows"])}'
        )

impala_irf = pd.concat(impala_irf_parts, ignore_index=True) if impala_irf_parts else pd.DataFrame()
bridge = pd.DataFrame()
print('=== Impala recompute IRF (mart-like perimeter) ===')
display(impala_irf)

# Compare Impala int to final_df int
if len(impala_irf):
    bridge = impala_irf.merge(
        cmp_irf[['report_month', 'int_component_lake', 'int_component_excel', 'commission_from_ops_lake']],
        on='report_month',
        how='left',
    )
    bridge['d_impala_minus_final_df_int'] = (
        pd.to_numeric(bridge['int_component_sum_all_rows'], errors='coerce')
        - pd.to_numeric(bridge['int_component_lake'], errors='coerce')
    )
    bridge['d_impala_alive_minus_excel'] = (
        pd.to_numeric(bridge['int_component_sum_alive_only'], errors='coerce')
        - pd.to_numeric(bridge['int_component_excel'], errors='coerce')
    )
    bridge['d_impala_all_minus_excel'] = (
        pd.to_numeric(bridge['int_component_sum_all_rows'], errors='coerce')
        - pd.to_numeric(bridge['int_component_excel'], errors='coerce')
    )
    print('=== Impala IRF vs final_df / Excel ===')
    display(bridge[[
        'report_month',
        'int_component_sum_all_rows', 'int_component_sum_alive_only',
        'int_component_lake', 'int_component_excel',
        'd_impala_minus_final_df_int', 'd_impala_all_minus_excel', 'd_impala_alive_minus_excel',
        'trx_with_multi_int_rows', 'avg_int_rows_per_trx',
        'fee_pos_rows', 'fee_neg_rows', 'trx_fee_sum_pos', 'trx_fee_sum_neg',
    ]])


# ---------- C) TOP agr: lake vs excel int in March + Impala fee for those n_agr ----------
ex_m = excel_agr_df.loc[excel_agr_df['report_month'] == '2026-03'][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum']
].rename(columns={
    'chod': 'chod_excel', 'commission_from_ops': 'ops_excel',
    'int_component': 'int_excel', 'trx_sum': 'trx_excel',
})
lk_m = lake_agr.loc[lake_agr['report_month'] == '2026-03'][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum']
].rename(columns={
    'chod': 'chod_lake', 'commission_from_ops': 'ops_lake',
    'int_component': 'int_lake', 'trx_sum': 'trx_lake',
})
gap = ex_m.merge(lk_m, on=['inn_key', 'agr_id_key'], how='inner')
for c in ['chod_excel', 'chod_lake', 'ops_excel', 'ops_lake', 'int_excel', 'int_lake', 'trx_excel', 'trx_lake']:
    gap[c] = pd.to_numeric(gap[c], errors='coerce')
gap['d_chod'] = gap['chod_lake'].fillna(0) - gap['chod_excel'].fillna(0)
gap['d_int'] = gap['int_lake'].fillna(0) - gap['int_excel'].fillna(0)
gap['d_ops'] = gap['ops_lake'].fillna(0) - gap['ops_excel'].fillna(0)
gap['int_explains_chod'] = (gap['d_int'].abs() >= gap['d_chod'].abs() * 0.8) & (gap['d_chod'] < 0)

top_gap = gap.sort_values('d_chod').head(25).copy()
print('=== TOP-25 March agr by d_chod (lake-excel); flag if d_int explains ===')
display(top_gap[[
    'inn_key', 'agr_id_key', 'chod_excel', 'chod_lake', 'd_chod',
    'int_excel', 'int_lake', 'd_int', 'ops_excel', 'ops_lake', 'd_ops',
    'trx_excel', 'trx_lake', 'int_explains_chod',
]])
print(
    f'Among TOP25: int_explains_chod={int(top_gap["int_explains_chod"].sum())}/25 | '
    f'sum d_chod={top_gap["d_chod"].sum():,.0f} | sum d_int={top_gap["d_int"].sum():,.0f}'
)

# Map agr_id -> n_agr for TOP gap (SA)
agr_ids = clean_keys(top_gap['agr_id_key'].astype(str).tolist()) if 'clean_keys' in dir() else sorted(set(
    str(x).strip() for x in top_gap['agr_id_key'].tolist() if pd.notna(x) and str(x).strip()
))

def _sql_in(vals):
    out = []
    for v in vals:
        s = str(v).strip().replace("'", "''")
        if s:
            out.append(f"'{s}'")
    return ', '.join(out) if out else "''"

ms, me = month_bounds('2026-03')
sql_top_agr_fee = f"""
with agr_map as (
  select distinct
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) in ({_sql_in(agr_ids)})
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base as (
  select cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
  group by cast(t.n_trx as string)
),
ta as (
  select cast(a.n_agr as string) as n_agr, cast(a.n_trx as string) as n_trx,
         max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join agr_map am on am.n_agr = cast(a.n_agr as string)
  where coalesce(a.ods_deleted_flg, '0') <> '1'
  group by cast(a.n_agr as string), cast(a.n_trx as string)
),
fee as (
  select
    ta.n_agr,
    ta.n_trx,
    ta.n_amt_tax,
    sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as fee_sum,
    count(i.n_trx) as int_rows
  from ta
  left join ods_alpha.scd1_trx_int i
    on cast(i.n_trx as string) = ta.n_trx
  group by ta.n_agr, ta.n_trx, ta.n_amt_tax
)
select
  am.agr_id as agr_id_key,
  f.n_agr,
  count(distinct f.n_trx) as trx_cnt,
  sum(f.n_amt_tax) as ops_impala,
  sum(f.fee_sum) as int_impala,
  sum(case when f.int_rows > 1 then 1 else 0 end) as trx_multi_int,
  avg(f.int_rows) as avg_int_rows,
  sum(case when f.fee_sum < 0 then 1 else 0 end) as trx_fee_neg,
  sum(case when f.fee_sum > 0 then 1 else 0 end) as trx_fee_pos
from fee f
join agr_map am on am.n_agr = f.n_agr
group by am.agr_id, f.n_agr
"""

with imp:
    imp.execute('set MEM_LIMIT=16g')
    top_impala = imp.fetch(sql_top_agr_fee)

if top_impala is not None and len(top_impala):
    top_impala['agr_id_key'] = top_impala['agr_id_key'].astype(str).str.strip()
    for c in ['ops_impala', 'int_impala', 'trx_cnt', 'trx_multi_int', 'avg_int_rows']:
        top_impala[c] = pd.to_numeric(top_impala[c], errors='coerce')
    top_join = top_gap.merge(top_impala, on='agr_id_key', how='left')
    top_join['d_int_impala_minus_excel'] = top_join['int_impala'].fillna(0) - top_join['int_excel'].fillna(0)
    top_join['d_int_lake_minus_impala'] = top_join['int_lake'].fillna(0) - top_join['int_impala'].fillna(0)
    print('=== TOP gap agr: Impala fee vs Excel/lake int (March) ===')
    display(top_join[[
        'inn_key', 'agr_id_key', 'n_agr',
        'int_excel', 'int_lake', 'int_impala',
        'd_int', 'd_int_impala_minus_excel', 'd_int_lake_minus_impala',
        'ops_excel', 'ops_lake', 'ops_impala',
        'trx_cnt', 'trx_multi_int', 'avg_int_rows',
    ]])
else:
    top_join = pd.DataFrame()
    print('WARNING: top agr Impala fee query returned empty')


# ---------- D) Feb vs Mar Impala delta on same metrics ----------
if len(impala_irf) >= 2:
    piv = impala_irf.set_index('report_month')
    print('=== Impala Mar − Feb (source-level) ===')
    if '2026-03' in piv.index and '2026-02' in piv.index:
        for c in [
            'commission_from_ops', 'int_component_sum_all_rows', 'int_component_sum_alive_only',
            'trx_with_acq_sa', 'trx_with_any_int', 'trx_with_multi_int_rows',
            'int_rows_total', 'fee_neg_rows', 'fee_pos_rows',
            'avg_fee_per_trx_with_int', 'avg_int_rows_per_trx',
        ]:
            if c in piv.columns:
                a = float(pd.to_numeric(piv.loc['2026-03', c], errors='coerce') or 0)
                b = float(pd.to_numeric(piv.loc['2026-02', c], errors='coerce') or 0)
                print(f'  d_{c}: {a - b:,.4f}  (Mar={a:,.4f} Feb={b:,.4f})')


# ---------- E) Verdict ----------
print('=== VERDICT 5e ===')
verdicts = []
if len(impala_irf):
    mar = impala_irf.loc[impala_irf['report_month'] == '2026-03']
    feb = impala_irf.loc[impala_irf['report_month'] == '2026-02']
    if len(mar) and len(feb):
        d_int = float(mar.iloc[0]['int_component_sum_all_rows']) - float(feb.iloc[0]['int_component_sum_all_rows'])
        d_multi = float(mar.iloc[0]['trx_with_multi_int_rows']) - float(feb.iloc[0]['trx_with_multi_int_rows'])
        d_avg_rows = float(mar.iloc[0]['avg_int_rows_per_trx'] or 0) - float(feb.iloc[0]['avg_int_rows_per_trx'] or 0)
        if abs(d_int) > 3_000_000:
            verdicts.append(f'Источник trx_int за март уже тяжелее Feb по fee (d_int≈{d_int:,.0f}) — не артефакт pandas.')
        if d_multi > 1000 or d_avg_rows > 0.2:
            verdicts.append(
                f'Рост multi-row int на trx (d_multi_trx={d_multi:,.0f}, d_avg_rows={d_avg_rows:.3f}) — возможен двойной sum(fee).'
            )
        else:
            verdicts.append('Массового взрыва multi-row int на trx не видно — скорее объём/величина fee, не дубли строк.')
        # sign mix
        neg_m = float(mar.iloc[0]['fee_neg_rows'] or 0)
        pos_m = float(mar.iloc[0]['fee_pos_rows'] or 0)
        neg_f = float(feb.iloc[0]['fee_neg_rows'] or 0)
        pos_f = float(feb.iloc[0]['fee_pos_rows'] or 0)
        verdicts.append(
            f'Знаковый микс fee rows: Mar neg={neg_m:,.0f}/pos={pos_m:,.0f} vs Feb neg={neg_f:,.0f}/pos={pos_f:,.0f} '
            f'(если pos внезапно вырос при том же знаке суммы — смотри отдельно).'
        )

if 'bridge' in locals() and bridge is not None and len(bridge):
    mbridge = bridge.loc[bridge['report_month'] == '2026-03']
    if len(mbridge):
        d_ff = float(mbridge.iloc[0]['d_impala_minus_final_df_int'] or 0)
        if abs(d_ff) < 100_000:
            verdicts.append('final_df.int ≈ Impala recompute → витрина честно перенесла «кривой» март из источника.')
        else:
            verdicts.append(
                f'final_df.int расходится с Impala recompute на {d_ff:,.0f} — баг агрегации витрины, не только сырьё.'
            )
        d_ex = float(mbridge.iloc[0]['d_impala_all_minus_excel'] or 0)
        verdicts.append(f'Impala IRF − Excel IRF (Mar) ≈ {d_ex:,.0f} (ожидаем ~ d_chod lake−excel).')

if not verdicts:
    verdicts.append('Недостаточно данных Impala — проверь коннект/очередь и перезапусти 5e.')

for v in verdicts:
    print('-', v)

# saves
out_dir = OUTPUT_DIR
cmp_irf.to_csv(out_dir / 'march_irf_lake_vs_excel_by_month.csv', index=False, encoding='utf-8-sig')
if len(impala_irf):
    impala_irf.to_csv(out_dir / 'march_irf_impala_perimeter_feb_mar_apr.csv', index=False, encoding='utf-8-sig')
if 'bridge' in locals() and bridge is not None and len(bridge):
    bridge.to_csv(out_dir / 'march_irf_impala_vs_final_df_bridge.csv', index=False, encoding='utf-8-sig')
top_gap.to_csv(out_dir / 'march_irf_top25_agr_lake_vs_excel.csv', index=False, encoding='utf-8-sig')
if len(top_join):
    top_join.to_csv(out_dir / 'march_irf_top25_agr_impala_fee.csv', index=False, encoding='utf-8-sig')
print('Saved IRF diagnostic CSVs under', out_dir)


## 5f) Multi-row `trx_int` в марте: дубли fee или легитимные части?

После 5e: в марте ~667k trx с **>1 строкой** в `scd1_trx_int` (Feb≈1, Apr≈0).

Проверяем:
1. `describe` / какие колонки есть у `trx_int` (ключ дубля).
2. На multi-int trx: совпадают ли `n_amt_fee` между строками (полные дубли).
3. Эффект агрегации на весь март: `sum(fee)` vs `max(fee)` vs `sum(distinct fee)` vs 1 row (`min`/`any`).
4. Насколько `max`-агрегация сближает IRF/ЧОД с Excel.
5. Sample 30 multi-int `n_trx` глазами + Feb control (ожидаем почти без multi).


In [ ]:
# 5f) Multi-row trx_int in March: duplicates vs legitimate parts
from IPython.display import display
from calendar import monthrange

if 'imp' not in globals() or imp is None:
    raise RuntimeError('Сначала 5e.0 (Impala connect)')

MONTH = '2026-03'
CONTROL = '2026-02'
y, m = map(int, MONTH.split('-'))
month_start = f'{MONTH}-01'
month_end = f'{MONTH}-{monthrange(y, m)[1]:02d}'
cy, cm = map(int, CONTROL.split('-'))
ctrl_start = f'{CONTROL}-01'
ctrl_end = f'{CONTROL}-{monthrange(cy, cm)[1]:02d}'


# ---------- 0) columns of trx_int ----------
print('=== describe ods_alpha.scd1_trx_int ===')
with imp:
    desc = imp.fetch('describe ods_alpha.scd1_trx_int')
display(desc)
colname = None
for c in desc.columns:
    if str(c).lower() in {'name', 'col_name', 'column_name', 'field'}:
        colname = c
        break
if colname is None:
    colname = desc.columns[0]
int_cols = [str(x).strip() for x in desc[colname].tolist()]
print('columns:', int_cols)

# pick useful attribute cols for dup fingerprint (besides n_trx / fee)
extra_cols = []
for cand in [
    'n_amt_fee', 'c_fee_type', 'c_fee_code', 'c_type', 'c_fee_name',
    'n_fee_type', 'c_irf_type', 'c_oper_type', 'd_fee', 'd_trx',
    'n_row', 'c_src', 'ods_deleted_flg', 'n_trx',
]:
    for c in int_cols:
        if c.lower() == cand.lower() and c not in extra_cols:
            extra_cols.append(c)
# always try a few present cols
for c in int_cols:
    cl = c.lower()
    if any(k in cl for k in ['fee', 'type', 'code', 'irf', 'oper', 'src', 'deleted']) and c not in extra_cols:
        extra_cols.append(c)
extra_cols = extra_cols[:12]
print('fingerprint cols:', extra_cols)


# ---------- 1) Core CTE: mart-like trx keys for a month ----------
def cte_trx_keys(ms, me):
    return f"""
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
      group by cast(t.n_trx as string)
    ),
    ta as (
      select cast(a.n_trx as string) as n_trx
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string)
    )
    """


# ---------- 2) Classification of multi-int trx ----------
sql_multi_class = f"""
with {cte_trx_keys(month_start, month_end)},
int_alive as (
  select
    cast(i.n_trx as string) as n_trx,
    coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
),
per_trx as (
  select
    n_trx,
    count(*) as int_rows,
    count(distinct cast(n_amt_fee as string)) as distinct_fee_values,
    sum(n_amt_fee) as fee_sum,
    max(n_amt_fee) as fee_max,
    min(n_amt_fee) as fee_min,
    avg(n_amt_fee) as fee_avg
  from int_alive
  group by n_trx
),
multi as (
  select *
  from per_trx
  where int_rows > 1
)
select
  count(*) as multi_trx_cnt,
  sum(int_rows) as multi_int_rows,
  sum(case when distinct_fee_values = 1 then 1 else 0 end) as multi_same_fee_dup,
  sum(case when distinct_fee_values > 1 then 1 else 0 end) as multi_different_fees,
  sum(case when distinct_fee_values = 1 then fee_sum else 0 end) as fee_sum_on_same_fee_dups,
  sum(case when distinct_fee_values = 1 then fee_max else 0 end) as fee_max_on_same_fee_dups,
  sum(case when distinct_fee_values = 1 then fee_sum - fee_max else 0 end) as extra_fee_from_same_fee_dups,
  sum(case when distinct_fee_values > 1 then fee_sum else 0 end) as fee_sum_on_different_fees,
  sum(case when distinct_fee_values > 1 then fee_max else 0 end) as fee_max_on_different_fees,
  avg(int_rows) as avg_rows_on_multi,
  avg(distinct_fee_values) as avg_distinct_fee_on_multi
from multi
"""

print(f'=== {MONTH}: classify multi-int trx (same fee = full duplicate?) ===')
with imp:
    imp.execute('set MEM_LIMIT=16g')
    multi_class = imp.fetch(sql_multi_class)
display(multi_class)

if multi_class is not None and len(multi_class):
    r = multi_class.iloc[0]
    same = float(r['multi_same_fee_dup'] or 0)
    diff = float(r['multi_different_fees'] or 0)
    extra = float(r['extra_fee_from_same_fee_dups'] or 0)
    print(
        f'same-fee dups: {same:,.0f} trx | different fees: {diff:,.0f} trx | '
        f'extra fee from same-fee dups (sum-max): {extra:,.2f}'
    )


# ---------- 3) Whole-month aggregation alternatives ----------
sql_agg_alts = f"""
with {cte_trx_keys(month_start, month_end)},
int_alive as (
  select
    cast(i.n_trx as string) as n_trx,
    coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
),
per_trx as (
  select
    n_trx,
    sum(n_amt_fee) as fee_sum,
    max(n_amt_fee) as fee_max,
    min(n_amt_fee) as fee_min,
    -- approximate distinct-sum via nested: sum of unique fee values (Impala)
    -- fallback: if all equal, = max; else keep sum (computed below in pandas if needed)
    count(*) as int_rows,
    count(distinct cast(n_amt_fee as string)) as distinct_fee_values
  from int_alive
  group by n_trx
)
select
  count(*) as trx_with_int,
  sum(fee_sum) as int_sum_all_rows,
  sum(fee_max) as int_max_per_trx,
  sum(fee_min) as int_min_per_trx,
  sum(case when distinct_fee_values = 1 then fee_max else fee_sum end) as int_dedupe_same_fee_else_sum,
  sum(case when int_rows > 1 then 1 else 0 end) as multi_trx,
  sum(case when int_rows > 1 then fee_sum - fee_max else 0 end) as extra_vs_max_on_multi
from per_trx
"""

print(f'=== {MONTH}: IRF aggregation alternatives (mart perimeter) ===')
with imp:
    imp.execute('set MEM_LIMIT=16g')
    agg_alts = imp.fetch(sql_agg_alts)
display(agg_alts)

# Control month
sql_agg_ctrl = f"""
with {cte_trx_keys(ctrl_start, ctrl_end)},
int_alive as (
  select
    cast(i.n_trx as string) as n_trx,
    coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
),
per_trx as (
  select
    n_trx,
    sum(n_amt_fee) as fee_sum,
    max(n_amt_fee) as fee_max,
    min(n_amt_fee) as fee_min,
    count(*) as int_rows,
    count(distinct cast(n_amt_fee as string)) as distinct_fee_values
  from int_alive
  group by n_trx
)
select
  count(*) as trx_with_int,
  sum(fee_sum) as int_sum_all_rows,
  sum(fee_max) as int_max_per_trx,
  sum(fee_min) as int_min_per_trx,
  sum(case when distinct_fee_values = 1 then fee_max else fee_sum end) as int_dedupe_same_fee_else_sum,
  sum(case when int_rows > 1 then 1 else 0 end) as multi_trx,
  sum(case when int_rows > 1 then fee_sum - fee_max else 0 end) as extra_vs_max_on_multi
from per_trx
"""
print(f'=== {CONTROL}: same aggregation alternatives (control) ===')
with imp:
    imp.execute('set MEM_LIMIT=16g')
    agg_ctrl = imp.fetch(sql_agg_ctrl)
display(agg_ctrl)


# Bridge to Excel / final_df
excel_int_mar = np.nan
lake_int_mar = np.nan
if 'excel_agr_df' in globals() and excel_agr_df is not None:
    excel_int_mar = float(
        pd.to_numeric(
            excel_agr_df.loc[excel_agr_df['report_month'] == MONTH, 'int_component'],
            errors='coerce',
        ).fillna(0).sum()
    )
if 'lake_agr' in globals() and lake_agr is not None:
    lake_int_mar = float(
        pd.to_numeric(
            lake_agr.loc[lake_agr['report_month'] == MONTH, 'int_component'],
            errors='coerce',
        ).fillna(0).sum()
    )

alt_cmp = None
if agg_alts is not None and len(agg_alts):
    a = agg_alts.iloc[0]
    alt_cmp = pd.DataFrame([
        {
            'method': 'sum_all_rows (current mart)',
            'int_total': float(a['int_sum_all_rows']),
            'delta_vs_excel': float(a['int_sum_all_rows']) - excel_int_mar if pd.notna(excel_int_mar) else np.nan,
            'delta_vs_final_df': float(a['int_sum_all_rows']) - lake_int_mar if pd.notna(lake_int_mar) else np.nan,
        },
        {
            'method': 'max_per_trx',
            'int_total': float(a['int_max_per_trx']),
            'delta_vs_excel': float(a['int_max_per_trx']) - excel_int_mar if pd.notna(excel_int_mar) else np.nan,
            'delta_vs_final_df': float(a['int_max_per_trx']) - lake_int_mar if pd.notna(lake_int_mar) else np.nan,
        },
        {
            'method': 'dedupe_same_fee_else_sum',
            'int_total': float(a['int_dedupe_same_fee_else_sum']),
            'delta_vs_excel': float(a['int_dedupe_same_fee_else_sum']) - excel_int_mar if pd.notna(excel_int_mar) else np.nan,
            'delta_vs_final_df': float(a['int_dedupe_same_fee_else_sum']) - lake_int_mar if pd.notna(lake_int_mar) else np.nan,
        },
        {
            'method': 'excel_march',
            'int_total': excel_int_mar,
            'delta_vs_excel': 0.0,
            'delta_vs_final_df': excel_int_mar - lake_int_mar if pd.notna(lake_int_mar) else np.nan,
        },
        {
            'method': 'final_df_march',
            'int_total': lake_int_mar,
            'delta_vs_excel': lake_int_mar - excel_int_mar if pd.notna(excel_int_mar) else np.nan,
            'delta_vs_final_df': 0.0,
        },
    ])
    print(f'=== IRF methods vs Excel/final_df ({MONTH}) ===')
    print(f'excel_int={excel_int_mar:,.2f} | final_df_int={lake_int_mar:,.2f}')
    display(alt_cmp)


# ---------- 4) Sample multi-int trx rows ----------
# dynamic select list
sel_cols = ['cast(i.n_trx as string) as n_trx', 'cast(i.n_amt_fee as double) as n_amt_fee']
for c in extra_cols:
    if c.lower() in {'n_trx', 'n_amt_fee'}:
        continue
    sel_cols.append(f'cast(i.`{c}` as string) as `{c}`')
sel_sql = ',\n      '.join(sel_cols)

sql_sample = f"""
with {cte_trx_keys(month_start, month_end)},
int_alive as (
  select
    cast(i.n_trx as string) as n_trx,
    coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
),
multi as (
  select n_trx
  from int_alive
  group by n_trx
  having count(*) > 1
),
picked as (
  select n_trx
  from multi
  order by n_trx
  limit 30
)
select
      {sel_sql}
from ods_alpha.scd1_trx_int i
join picked p on p.n_trx = cast(i.n_trx as string)
where coalesce(i.ods_deleted_flg, '0') <> '1'
order by cast(i.n_trx as string), cast(i.n_amt_fee as double)
"""

print(f'=== Sample 30 multi-int n_trx raw rows ({MONTH}) ===')
with imp:
    imp.execute('set MEM_LIMIT=16g')
    try:
        sample_rows = imp.fetch(sql_sample)
    except Exception as exc:
        print('full sample failed, fallback minimal cols:', type(exc).__name__, str(exc)[:200])
        sql_sample_min = f"""
        with {cte_trx_keys(month_start, month_end)},
        int_alive as (
          select cast(i.n_trx as string) as n_trx, coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
          from ods_alpha.scd1_trx_int i
          join ta k on k.n_trx = cast(i.n_trx as string)
          where coalesce(i.ods_deleted_flg, '0') <> '1'
        ),
        multi as (
          select n_trx from int_alive group by n_trx having count(*) > 1
        ),
        picked as (select n_trx from multi order by n_trx limit 30)
        select cast(i.n_trx as string) as n_trx, cast(i.n_amt_fee as double) as n_amt_fee,
               cast(i.ods_deleted_flg as string) as ods_deleted_flg
        from ods_alpha.scd1_trx_int i
        join picked p on p.n_trx = cast(i.n_trx as string)
        order by 1, 2
        """
        sample_rows = imp.fetch(sql_sample_min)
display(sample_rows.head(80) if sample_rows is not None else sample_rows)

# Per-sample summary
if sample_rows is not None and len(sample_rows):
    s = sample_rows.copy()
    s['n_amt_fee'] = pd.to_numeric(s['n_amt_fee'], errors='coerce')
    samp_sum = (
        s.groupby('n_trx', as_index=False)
        .agg(
            int_rows=('n_amt_fee', 'size'),
            distinct_fee=('n_amt_fee', 'nunique'),
            fee_sum=('n_amt_fee', 'sum'),
            fee_max=('n_amt_fee', 'max'),
            fee_min=('n_amt_fee', 'min'),
        )
    )
    samp_sum['is_same_fee_dup'] = samp_sum['distinct_fee'] == 1
    samp_sum['extra_vs_max'] = samp_sum['fee_sum'] - samp_sum['fee_max']
    print('=== Sample multi-trx summary ===')
    display(samp_sum)
    print(
        f'sample same-fee dups: {int(samp_sum["is_same_fee_dup"].sum())}/'
        f'{len(samp_sum)} | extra_vs_max sum={samp_sum["extra_vs_max"].sum():,.2f}'
    )


# ---------- 5) Verdict ----------
print('=== VERDICT 5f ===')
if multi_class is not None and len(multi_class) and alt_cmp is not None:
    same = float(multi_class.iloc[0]['multi_same_fee_dup'] or 0)
    diff = float(multi_class.iloc[0]['multi_different_fees'] or 0)
    total_m = same + diff
    same_share = same / total_m if total_m else np.nan
    extra = float(multi_class.iloc[0]['extra_fee_from_same_fee_dups'] or 0)
    max_delta = float(alt_cmp.loc[alt_cmp['method'] == 'max_per_trx', 'delta_vs_excel'].iloc[0])
    sum_delta = float(alt_cmp.loc[alt_cmp['method'] == 'sum_all_rows (current mart)', 'delta_vs_excel'].iloc[0])
    dedupe_delta = float(alt_cmp.loc[alt_cmp['method'] == 'dedupe_same_fee_else_sum', 'delta_vs_excel'].iloc[0])

    print(f'Multi-int trx: same-fee={same:,.0f} ({same_share:.1%}), different-fee={diff:,.0f}')
    print(f'Extra IRF from same-fee dups (sum-max): {extra:,.2f}')
    print(f'delta vs Excel: sum={sum_delta:,.0f} | max_per_trx={max_delta:,.0f} | dedupe_same={dedupe_delta:,.0f}')

    if same_share >= 0.8 and abs(extra) >= 3_000_000:
        print(
            'SUPPORT: в марте доминируют ПОЛНЫЕ дубли fee на trx. '
            'Текущий sum(n_amt_fee) раздувает IRF. Кандидат фикса: max(fee) или dedupe одинаковых fee.'
        )
    elif same_share >= 0.5:
        print(
            'PARTIAL: много same-fee дублей, но есть и разные fee. '
            'Нужно правило: dedupe identical rows, sum разных типов fee (если есть тип в колонках).'
        )
    else:
        print(
            'WEAK на полных дублях: чаще разные fee на trx. '
            'Тогда Excel, возможно, берёт одну строку/тип; смотри sample колонки типа fee.'
        )

    if abs(max_delta) < abs(sum_delta) * 0.4:
        print('max_per_trx заметно ближе к Excel, чем sum — практичный hotfix для марта/всегда.')
    if abs(dedupe_delta) <= abs(max_delta):
        print('dedupe_same_fee_else_sum не хуже max — предпочтительнее, если бывают разные легитимные части.')
else:
    print('Не удалось получить multi_class/alt_cmp — проверь Impala.')

# saves
out = OUTPUT_DIR
if multi_class is not None:
    multi_class.to_csv(out / 'march_2026_multi_int_classification.csv', index=False, encoding='utf-8-sig')
if agg_alts is not None:
    agg_alts.to_csv(out / 'march_2026_irf_agg_alternatives.csv', index=False, encoding='utf-8-sig')
if agg_ctrl is not None:
    agg_ctrl.to_csv(out / 'feb_2026_irf_agg_alternatives.csv', index=False, encoding='utf-8-sig')
if alt_cmp is not None:
    alt_cmp.to_csv(out / 'march_2026_irf_methods_vs_excel.csv', index=False, encoding='utf-8-sig')
if sample_rows is not None:
    sample_rows.to_csv(out / 'march_2026_multi_int_sample_rows.csv', index=False, encoding='utf-8-sig')
print('Saved 5f CSVs under', out)


## 6) Сохранение + вердикт


In [ ]:
out_monthly = OUTPUT_DIR / 'excel_monthly_chod_finrez_2026_01_2026_06.csv'
out_march_peers = OUTPUT_DIR / 'excel_march_2026_vs_peers.csv'
out_top_chod = OUTPUT_DIR / 'excel_march_2026_top_chod_drop_vs_feb.csv'
out_comp = OUTPUT_DIR / 'excel_march_2026_component_deltas_vs_feb.csv'

monthly_excel.to_csv(out_monthly, index=False, encoding='utf-8-sig')
march_vs_peers.to_csv(out_march_peers, index=False, encoding='utf-8-sig')
top_chod_feb.to_csv(out_top_chod, index=False, encoding='utf-8-sig')
comp_top.to_csv(out_comp, index=False, encoding='utf-8-sig')

print('Saved:')
print(' ', out_monthly)
print(' ', out_march_peers)
print(' ', out_top_chod)
print(' ', out_comp)

# Auto verdict
chod_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'chod', 'pct_vs_peer_mean'].iloc[0])
fin_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'fin_result', 'pct_vs_peer_mean'].iloc[0])
ops_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'commission_from_ops', 'pct_vs_peer_mean'].iloc[0])
trx_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'trx_sum', 'pct_vs_peer_mean'].iloc[0])
aur_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'aur', 'pct_vs_peer_mean'].iloc[0])

print('=== VERDICT (Excel-only) ===')
print(f'March CHOD vs peer mean: {chod_pct*100:.1f}%')
print(f'March fin_result vs peer mean: {fin_pct*100:.1f}%')
print(f'March commission_from_ops vs peer mean: {ops_pct*100:.1f}%')
print(f'March trx_sum vs peer mean: {trx_pct*100:.1f}%')
print(f'March AUR vs peer mean: {aur_pct*100:.1f}%')

drivers = []
if pd.notna(ops_pct) and ops_pct < -0.03:
    drivers.append('commission_from_ops')
if pd.notna(trx_pct) and trx_pct < -0.03:
    drivers.append('trx_sum')
if pd.notna(aur_pct) and aur_pct > 0.03:
    drivers.append('AUR higher (worsens finrez)')
if len(only_feb_not_mar) > 50:
    drivers.append(f'missing agrs vs Feb ({len(only_feb_not_mar)})')

if not drivers:
    drivers.append('see TOP agr deltas / identity gap')

print('Likely drivers:', ', '.join(drivers))
print(
    'Finrez follows CHOD if AUR/amort stable; if finrez drops more than CHOD — check AUR/amort uptick.'
)
